# Notebook 04 — Matryoshka Representation Learning (MRL)

MRL trains embeddings so that the first N dimensions of a large embedding are as informative as a fully-trained N-dimensional model. This lets you trade off speed vs quality at inference time.

In [ ]:
# !pip install sentence-transformers datasets

## 1. Understand the concept

In [ ]:
# Matryoshka embeddings: a 768-dim embedding where the first 64 dims
# are already high quality, allowing you to truncate at inference time.

# Training objective: sum of losses at multiple dimensionalities
# Loss = sum_d in {64, 128, 256, 768} of CosineSimilarityLoss(embedding[:d])

import numpy as np

# Simulated: show how truncation affects similarity
np.random.seed(42)
emb_a = np.random.randn(768)
emb_b = emb_a + np.random.randn(768) * 0.1  # nearly identical

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

for dim in [32, 64, 128, 256, 512, 768]:
    sim = cosine(emb_a[:dim], emb_b[:dim])
    print(f"dim={dim:4d}: similarity={sim:.4f}")

## 2. Train with MRL loss

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.losses import MatryoshkaLoss
from torch.utils.data import DataLoader
from datasets import load_dataset

model = SentenceTransformer("all-MiniLM-L6-v2")

dataset = load_dataset("sentence-transformers/all-nli", "pair", split="train[:2000]")
train_examples = [
    InputExample(texts=[row["sentence1"], row["sentence2"]], label=float(row["label"]))
    for row in dataset
]
dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

inner_loss = losses.CosineSimilarityLoss(model)
mrl_loss = MatryoshkaLoss(model, inner_loss, matryoshka_dims=[32, 64, 128, 256, 384])

model.fit(
    train_objectives=[(dataloader, mrl_loss)],
    epochs=1,
    warmup_steps=50,
    output_path="./mrl_model",
)
print("MRL training complete.")

## 3. Inference — choose your dimensionality

In [ ]:
model_loaded = SentenceTransformer("./mrl_model")

sentences = ["The quick brown fox", "A fast auburn vulpine"]
full_embs = model_loaded.encode(sentences)
print("Full embedding shape:", full_embs.shape)

# Truncate to 64 dims for 12x faster ANN search
small_embs = full_embs[:, :64]
print("Truncated to 64 dims:", small_embs.shape)